# Step 4 — Extract Entities & Relationships with the LLM

So far our pipeline has been *deterministic*: clean and chunk are just rules. Now we do the one thing rules can't — **read prose and pull out structured knowledge**.

```
chunk text  --LLM-->  { entities: [...], relationships: [...] }   (validated by Pydantic)
```

For every chunk we ask the LLM: *what things are mentioned, and how are they connected?* The answer — vulnerabilities, products, agencies, standards, and the links between them (`AFFECTS`, `ISSUED_BY`, `MITIGATES`, ...) — becomes the **knowledge graph** we load into Neo4j in Step 5.

This notebook covers:
1. Why we use an LLM here (and only here)
2. **Structured output** via a Pydantic schema (no fragile text parsing)
3. **Prompt design** and a **controlled vocabulary**
4. **Grounding** techniques that keep the model from hallucinating
5. The **Groq-primary / Gemini-fallback** setup

> Code lives in `signalpulse/extraction.py` and `signalpulse/llm.py`.

## 1. Concepts

**Entities & relationships.** An *entity* is a "thing" the text talks about (an agency, a product, a CVE). A *relationship* is a stated link between two entities, written as a triple `(source) -RELATION-> (target)`, e.g. `(CVE-2026-48908) -AFFECTS-> (SP Page Builder)`. A pile of these triples *is* a knowledge graph.

**Why an LLM here (and only here)?** Extracting facts from free-flowing prose is exactly what LLMs are good at and what rules are bad at. Cleaning/chunking stay deterministic (cheap, repeatable); the LLM is reserved for this genuinely hard step.

**Structured output (the important part).** We do **not** ask for free text and then parse it. We define a **Pydantic** schema (`Extraction`) and call LangChain's `with_structured_output(Extraction)`. Under the hood this uses the provider's *function-calling / JSON mode*, which **forces** the model to return data matching our schema. Benefits:
- No regex, no "Sure! Here is your JSON:" preambles to strip.
- Invalid output is rejected/validated by Pydantic automatically.
- The field descriptions in the schema double as instructions to the model.

**Controlled vocabulary.** Entity `type` is an `Enum` (Organization, Product, Vulnerability, Regulation, Standard, ...) and relations are guided to a small `UPPER_SNAKE_CASE` set. This stops the graph from fragmenting into thousands of near-duplicate labels.

**Grounding (anti-hallucination).** Four safeguards:
1. **Temperature 0** — deterministic, no "creative" invention.
2. **Explicit instructions** — "use ONLY the passage; inventing facts is forbidden; an empty result is correct."
3. **Empty is allowed** — the model isn't pressured to produce output for boilerplate.
4. **Post-validation** — in code we drop any relationship whose endpoints weren't themselves extracted as entities (a classic hallucination signature) and de-duplicate.

**Groq primary, Gemini fallback.** Gemini's free tier allows only ~20 requests/day for `gemini-3.5-flash` — far too few for extracting over many chunks. So **Groq** (`openai/gpt-oss-120b`, much higher free limits, strong tool-calling) is the primary, and Gemini is the automatic fallback via `.with_fallbacks(...)`.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import llm
from signalpulse import extraction as X

# Provider order (primary first). Groq should lead; Gemini is the fallback.
print("LLM providers (primary first):", llm.available_providers())

LLM providers (primary first): ['groq', 'gemini']


## 2. The schema we force the LLM to fill

`Extraction` (in `extraction.py`) is the Pydantic model the LLM must return. Its
JSON Schema — including the allowed entity types and every field description — is
sent to the provider as the function/tool definition. Let's look at it.

In [2]:
import json

print("Allowed entity types:")
print("  " + ", ".join(t.value for t in X.EntityType))

print("\nExtraction JSON schema (what the model is constrained to):")
print(json.dumps(X.Extraction.model_json_schema(), indent=2)[:1400], "...")

Allowed entity types:
  Organization, Product, Vulnerability, Regulation, Standard, Technology, Person, Location, Event, Other

Extraction JSON schema (what the model is constrained to):
{
  "$defs": {
    "Entity": {
      "properties": {
        "name": {
          "description": "The entity's name, copied exactly as it appears in the passage.",
          "title": "Name",
          "type": "string"
        },
        "type": {
          "description": "The single best category for this entity. Choose EXACTLY ONE of: Organization, Product, Vulnerability, Regulation, Standard, Technology, Person, Location, Event. Use 'Other' if none fit.",
          "title": "Type",
          "type": "string"
        }
      },
      "required": [
        "name",
        "type"
      ],
      "title": "Entity",
      "type": "object"
    },
    "Relationship": {
      "properties": {
        "source": {
          "description": "Name of the source entity. MUST match one of the extracted entity names.",

## 3. The prompt

The system prompt sets the extraction rules (grounding, controlled relations, empty-is-ok). The human prompt supplies the document title and the passage. Here's the exact system prompt.

In [3]:
print(X.SYSTEM_PROMPT)

You are an information-extraction engine for a public-sector intelligence system. You read one passage of text and extract a knowledge graph of entities and relationships from it.

Strict rules:
1. GROUNDING: Extract ONLY facts explicitly stated in the passage. Never use outside knowledge, and never guess or infer facts that are not written.
2. If the passage contains no meaningful entities, return empty lists. An empty result is correct and expected for boilerplate or unrelated text.
3. Entity names must be copied from the passage (you may trim surrounding words, but do not rephrase or expand acronyms unless the passage does).
4. Choose the single best `type` for each entity from the allowed categories.
5. Create a relationship whenever the passage states a connection between two of your extracted entities -- for example a vulnerability in/affecting a product (AFFECTS), a rule or patch issued by an organization (ISSUED_BY), or a control that mitigates a risk (MITIGATES). Both entities

## 4. Extract from a real passage

Let's run the extractor on a CISA-style vulnerability advisory and print the triples. `extract_from_text` returns a validated, de-duplicated `Extraction`.

In [4]:
passage = (
    "CISA added CVE-2026-48908, a vulnerability in JoomShaper SP Page Builder, "
    "to its Known Exploited Vulnerabilities catalog. The flaw allows attackers to "
    "upload dangerous files and execute code. Vendors should apply the patches "
    "issued by JoomShaper before the federal remediation deadline."
)

result = X.extract_from_text(passage, title="CISA KEV Advisory")

print("ENTITIES")
for e in result.entities:
    print(f"  - {e.name:40s} [{e.type}]")

print("\nRELATIONSHIPS (graph triples)")
for r in result.relationships:
    print(f"  ({r.source}) -{r.relation}-> ({r.target})")

C:\Users\saihaj\Documents\22nd Project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:49: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


ENTITIES
  - CISA                                     [Organization]
  - CVE-2026-48908                           [Vulnerability]
  - JoomShaper SP Page Builder               [Product]
  - JoomShaper                               [Organization]

RELATIONSHIPS (graph triples)
  (CVE-2026-48908) -AFFECTS-> (JoomShaper SP Page Builder)


## 5. Grounding in action

Two checks that the model isn't making things up:

- **Unrelated text** → it should return *nothing* (empty lists are correct).
- **Post-validation** → if the model ever emits a relationship pointing at an entity it didn't extract (a hallucination tell), `_clean()` drops it. We simulate that below.

In [5]:
# (a) Unrelated / boilerplate text -> expect empty extraction
noise = X.extract_from_text(
    "The office cafeteria will be closed on Friday for routine maintenance.",
    title="Facilities Notice",
)
print(f"(a) unrelated text -> {len(noise.entities)} entities, "
      f"{len(noise.relationships)} relationships  (want 0/0)")

# (b) Simulate a hallucinated relationship and show post-validation removing it
from signalpulse.extraction import Extraction, Entity, Relationship, _clean

dirty = Extraction(
    entities=[
        Entity(name="CISA", type="Organization"),
        Entity(name="CVE-2026-48908", type="Vulnerability"),
    ],
    relationships=[
        Relationship(source="CVE-2026-48908", relation="reported by", target="CISA"),   # valid
        Relationship(source="CVE-2026-48908", relation="AFFECTS", target="Acme Router"), # dangling -> drop
    ],
)
cleaned = _clean(dirty)
print(f"\n(b) before cleaning: {len(dirty.relationships)} relationships")
for r in cleaned.relationships:
    print(f"    kept: ({r.source}) -{r.relation}-> ({r.target})")
print(f"    dropped 'Acme Router' triple (target was never extracted as an entity)")
print(f"    note: relation normalized to UPPER_SNAKE_CASE -> {cleaned.relationships[0].relation}")

(a) unrelated text -> 1 entities, 0 relationships  (want 0/0)

(b) before cleaning: 2 relationships
    kept: (CVE-2026-48908) -REPORTED_BY-> (CISA)
    dropped 'Acme Router' triple (target was never extracted as an entity)
    note: relation normalized to UPPER_SNAKE_CASE -> REPORTED_BY


## 6. Extract over real fetched + chunked documents

Now the end-to-end path: fetch documents (Step 2) → clean/chunk (Step 3) → extract (Step 4). To respect free-tier limits we only run the LLM over the first few chunks here. We then aggregate all triples into one small graph preview.

In [6]:
from signalpulse import connectors as C
from signalpulse import processing as P

# Fetch two real documents and chunk them.
docs = C.CISAKEVConnector().fetch(limit=2)
all_chunks = []
for d in docs:
    for ch in P.split_document(d):        # embeddings not needed for extraction
        all_chunks.append((d, ch))

# Keep this small to stay well within free-tier limits.
sample_chunks = all_chunks[:4]
print(f"Running extraction over {len(sample_chunks)} chunks...\n")

entities, triples = {}, set()
for doc, ch in sample_chunks:
    ex = X.extract_from_text(ch.text, title=doc.title)
    for e in ex.entities:
        entities[(e.name.lower())] = (e.name, e.type)
    for r in ex.relationships:
        triples.add((r.source, r.relation, r.target))
    print(f"  {ch.id:24s} -> {len(ex.entities)} entities, {len(ex.relationships)} relationships")

print(f"\nAggregated: {len(entities)} unique entities, {len(triples)} unique relationships")

Running extraction over 4 chunks...



  CVE-2026-48908::0        -> 4 entities, 2 relationships


  CVE-2026-48908::1        -> 2 entities, 0 relationships


  CVE-2026-55255::0        -> 5 entities, 2 relationships


  CVE-2026-55255::1        -> 1 entities, 0 relationships

Aggregated: 8 unique entities, 4 unique relationships


In [7]:
print("Entities extracted from the sample:")
for name, typ in sorted(entities.values(), key=lambda x: x[1]):
    print(f"  - {name:45s} [{typ}]")

print("\nGraph triples (this is what becomes the knowledge graph in Step 5):")
for s, rel, t in sorted(triples):
    print(f"  ({s}) -{rel}-> ({t})")

Entities extracted from the sample:
  - JoomShaper                                    [Organization]
  - CISA                                          [Organization]
  - Stakeholders                                  [Other]
  - SP Page Builder                               [Product]
  - Langflow                                      [Product]
  - BOD 26-04                                     [Regulation]
  - CVE-2026-48908                                [Vulnerability]
  - CVE-2026-55255                                [Vulnerability]

Graph triples (this is what becomes the knowledge graph in Step 5):
  (CISA) -ISSUED_BY-> (BOD 26-04)
  (CVE-2026-48908) -AFFECTS-> (SP Page Builder)
  (CVE-2026-55255) -AFFECTS-> (Langflow)
  (JoomShaper) -ISSUED_BY-> (SP Page Builder)


## Recap & what's next

We built the **knowledge-extraction** stage:

- **Structured output** — a Pydantic `Extraction` schema + `with_structured_output` gives us validated JSON, no parsing.
- **Controlled vocabulary** — enum entity types and a small relation set keep the graph consistent.
- **Grounding** — temperature 0, explicit "only-the-passage / empty-is-ok" rules, and a post-validation pass that drops dangling (hallucinated) relationships.
- **Groq primary, Gemini fallback** — high free throughput with automatic failover.

Each chunk now yields entities and triples like `(CVE) -AFFECTS-> (Product)`.

**Next — Step 5:** load everything into Neo4j — `Document` and `Chunk` nodes (with their embeddings), plus the extracted `Entity` nodes and relationships — **idempotently**, so re-running the pipeline never creates duplicates.

> Prompt to continue:
> *"Step 5: Create notebook `05_load_graph.ipynb`. Write the loader that upserts Documents, Chunks (with embeddings), and extracted Entities/relationships into Neo4j idempotently using MERGE, and link chunks to their document and to the entities they mention. Show the resulting graph counts and a sample query."*